# Stage 1 — Dataset Audit Runbook

Colab entry point for Stage 1 (`CLAUDE.md` roadmap). Runs the FoR-2sec full audit and the
DEEP-VOICE metadata-only audit (decision D6), and writes `docs/dataset_audit.md`'s evidence.

**Before running:** edit `configs/audit.yaml` in your checked-out copy of the repo with the
real paths for this session (or leave the Colab defaults if they match). This notebook never
hardcodes a Drive/Colab path — every path used below comes from that file.

This notebook only runs the audit tools in `src/audit/`. It does not do any preprocessing,
model, training, or evaluation work — that belongs to later stages.

## 1. Mount Drive, sync the repo, install, run tests

Mounts Google Drive, clones the repo on first run (or pulls latest on later runs), installs
`requirements.txt`, and runs the full test suite. **Stops here if any test fails** — an audit
run on top of a broken tool is not trustworthy evidence.

In [ ]:
import subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "<git remote URL>"  # fill in once the GitHub repo exists (Exec Plan 0.3)
REPO_DIR = Path("/content/voice-spoofing-detection")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd {REPO_DIR}

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

test_result = subprocess.run([sys.executable, "-m", "pytest", "-q"])
if test_result.returncode != 0:
    raise RuntimeError("pytest failed — fix before running the audit. Stopping.")

## 2. Download FoR-2sec

Resolves the exact Hugging Face dataset revision (commit sha) with `HfApi`, then downloads that
exact snapshot into the shared Drive `raw/` folder (download once, to Drive — never straight to
Colab's ephemeral disk), and copies it to local disk for fast I/O during the audit. Prints the
sha so it can be pasted into `docs/dataset_audit.md` §1 for traceability.

The HF viewer for this dataset has been observed to list a parquet-only, soundfolder-converted
snapshot for some revisions — if that's what a given revision resolves to, there are no raw audio
files on disk and every downstream step would silently operate on nothing. So this **stops with
an error if no audio files are found** after the snapshot download, rather than proceeding.

In [ ]:
import shutil
import yaml
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

cfg = yaml.safe_load(Path("configs/audit.yaml").read_text())

sha = HfApi().dataset_info("UncovAI/FOR-2sec").sha
print(f"FoR-2sec revision (HF commit sha): {sha}")

drive_for2sec = Path(cfg["drive"]["raw_root"]) / "for_2sec"
snapshot_download(repo_id="UncovAI/FOR-2sec", repo_type="dataset", revision=sha,
                   local_dir=str(drive_for2sec))

local_for2sec = Path(cfg["for2sec"]["root"])
local_for2sec.mkdir(parents=True, exist_ok=True)
shutil.copytree(drive_for2sec, local_for2sec, dirs_exist_ok=True)

audio_exts = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus"}
n_audio = sum(1 for p in local_for2sec.rglob("*") if p.suffix.lower() in audio_exts)
if n_audio == 0:
    raise RuntimeError(
        f"No audio files found under {local_for2sec} for revision {sha}. "
        "This revision may be a parquet-only snapshot — check the HF dataset viewer "
        "before proceeding.")
print(f"Found {n_audio} audio files under {local_for2sec}")

## 3. Download DEEP-VOICE

Downloads the Kaggle archive without unzipping first, so its sha256 can be recorded against the
exact zip that was pulled (not the extracted contents, which unzip tooling could alter). Then
unzips to local disk for the metadata-only audit.

**Reminder (hard rule 1, decision D6):** record the Kaggle dataset version number printed below
into `docs/dataset_audit.md` §1. Do not open, play, or plot any DEEP-VOICE file, at any point,
for any reason — the only permitted access is `src/audit/deepvoice_metadata_audit.py`, which
reads headers and hashes only and actively blocks decoding while it runs.

In [ ]:
import hashlib, subprocess, zipfile
from pathlib import Path

DATASET_SLUG = "birdy654/deep-voice-deepfake-voice-recognition"
drive_deepvoice_dir = Path(cfg["drive"]["raw_root"]) / "deep_voice"
drive_deepvoice_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(["kaggle", "datasets", "download", "-d", DATASET_SLUG,
                "-p", str(drive_deepvoice_dir)], check=True)

zip_path = next(drive_deepvoice_dir.glob("*.zip"))
sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print(f"DEEP-VOICE archive: {zip_path.name}")
print(f"sha256: {sha256}")
print("Record the Kaggle dataset version number (from the dataset's Kaggle page) in "
      "docs/dataset_audit.md \u00a71 alongside this hash.")

local_deepvoice = Path(cfg["deepvoice"]["root"])
local_deepvoice.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(local_deepvoice)
print(f"Extracted to {local_deepvoice}")

## 4. Dry run: FoR-2sec audit on a small sample

Runs the full audit pipeline on the first 500 files only, so any bug or unexpected packaging
quirk surfaces in seconds/minutes rather than after a full multi-thousand-file run.

In [ ]:
subprocess.run([sys.executable, "-m", "src.audit.for2sec_audit",
                "--root", cfg["for2sec"]["root"],
                "--out", cfg["for2sec"]["out"] + "_dryrun",
                "--revision", sha,
                "--limit", "500"], check=True)

## 5. Full FoR-2sec audit

Same command, no `--limit`. Writes the per-file inventory, duplicate/near-duplicate tables, and
the machine-readable summary that `docs/dataset_audit.md` \u00a72\u20134 draws its evidence from.

In [ ]:
subprocess.run([sys.executable, "-m", "src.audit.for2sec_audit",
                "--root", cfg["for2sec"]["root"],
                "--out", cfg["for2sec"]["out"],
                "--revision", sha], check=True)

## 6. DEEP-VOICE metadata audit

Metadata-only (decision D6): file counts, headers, hashes, filenames, CSV schema. The module
itself blocks sample-decoding APIs for the duration of the run and raises if anything tries to
use them, so a violation cannot happen silently.

In [ ]:
subprocess.run([sys.executable, "-m", "src.audit.deepvoice_metadata_audit",
                "--root", cfg["deepvoice"]["root"],
                "--out", cfg["deepvoice"]["out"],
                "--revision", sha256], check=True)

## 7. Cross-dataset byte-hash overlap check

Compares `sha256_bytes` between the two inventories. FoR-2sec and DEEP-VOICE are unrelated
releases, so the expected overlap is exactly 0 — a nonzero count would mean the same file bytes
exist in both the training pool and the frozen external test set, which would silently break the
generalization claim the whole project rests on (hard rule 1). This is a check, not a fix: any
overlap found here is reported in `docs/dataset_audit.md` \u00a75, not resolved automatically.

In [ ]:
import pandas as pd

for2sec_inv = pd.read_csv(Path(cfg["for2sec"]["out"]) / "for2sec_inventory.csv")
deepvoice_inv = pd.read_csv(Path(cfg["deepvoice"]["out"]) / "deepvoice_inventory.csv")

overlap = set(for2sec_inv["sha256_bytes"]) & set(deepvoice_inv["sha256_bytes"])
print(f"Cross-dataset byte-hash overlap: {len(overlap)} file(s) (expected 0)")
if overlap:
    print("Overlapping hashes (report these in docs/dataset_audit.md \u00a75, do not silently drop):")
    for h in overlap:
        print(" ", h)

## 8. Archive outputs to Drive

Copies `audit_out/` to the shared Drive so the raw evidence behind `docs/dataset_audit.md`
survives past this Colab session and is available to the rest of the team.

In [ ]:
archive_root = Path(cfg["drive"]["archive_root"])
archive_root.mkdir(parents=True, exist_ok=True)

for out_dir in (cfg["for2sec"]["out"], cfg["deepvoice"]["out"]):
    src = Path(out_dir)
    dst = archive_root / src.name
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Copied {src} -> {dst}")